# 02 — Data Processing
### Kenya Clean Cooking Access Gap Analysis

This notebook cleans, merges, and engineers the county-level master dataset that all subsequent notebooks depend on.

**Input:** Raw datasets from `data/raw/` and `data/shapefiles/`  
**Output:** `data/processed/kenya_county_clean_cooking.gpkg` (GeoPackage — shapefile + attributes)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import pyreadstat 

ROOT      = Path('..').resolve()
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
SHAPES    = ROOT / 'data' / 'shapefiles'
PROCESSED.mkdir(exist_ok=True)

---
## 1. Aggregate DHS Data to County Level

In [2]:
CLEAN_FUEL_CODES = [1, 2, 3]  # WHO: Electricity, LPG, Biogas

# DHS county codes map to Kenya's 47 counties
# HV024 in DHS Kenya 2022 maps to county code (1–47)
# Reference: KDHS 2022 appendix for county code lookup

COUNTY_CODES = {
    1: 'Mombasa', 2: 'Kwale', 3: 'Kilifi', 4: 'Tana River', 5: 'Lamu',
    6: 'Taita Taveta', 7: 'Garissa', 8: 'Wajir', 9: 'Mandera', 10: 'Marsabit',
    11: 'Isiolo', 12: 'Meru', 13: 'Tharaka-Nithi', 14: 'Embu', 15: 'Kitui',
    16: 'Machakos', 17: 'Makueni', 18: 'Nyandarua', 19: 'Nyeri', 20: 'Kirinyaga',
    21: "Murang'a", 22: 'Kiambu', 23: 'Turkana', 24: 'West Pokot', 25: 'Samburu',
    26: 'Trans Nzoia', 27: 'Uasin Gishu', 28: 'Elgeyo-Marakwet', 29: 'Nandi', 30: 'Baringo',
    31: 'Laikipia', 32: 'Nakuru', 33: 'Narok', 34: 'Kajiado', 35: 'Kericho',
    36: 'Bomet', 37: 'Kakamega', 38: 'Vihiga', 39: 'Bungoma', 40: 'Busia',
    41: 'Siaya', 42: 'Kisumu', 43: 'Homa Bay', 44: 'Migori', 45: 'Kisii',
    46: 'Nyamira', 47: 'Nairobi'
}

dhs_path = RAW / 'KEHR8CFL.dta'



if dhs_path.exists():
    df_dhs, meta = pyreadstat.read_dta(dhs_path)
    df_dhs.columns = df_dhs.columns.str.upper()
    df_dhs['clean_cooking']  = df_dhs['HV226'].isin(CLEAN_FUEL_CODES).astype(int)
    df_dhs['female_headed']  = (df_dhs['HV219'] == 2).astype(int)
    df_dhs['urban']          = (df_dhs['HV025'] == 1).astype(int)
    df_dhs['weight']         = df_dhs['HV005'] / 1_000_000  # DHS weight scaling
    df_dhs['county_name']    = df_dhs['HV024'].map(COUNTY_CODES)
    
    # Weighted aggregation to county level
    def weighted_mean(group, col, weight_col='weight'):
        return np.average(group[col], weights=group[weight_col])
    
    county_agg = df_dhs.groupby('county_name').apply(
        lambda g: pd.Series({
            'clean_cooking_rate':   weighted_mean(g, 'clean_cooking'),
            'female_headed_rate':   weighted_mean(g, 'female_headed'),
            'urban_rate':           weighted_mean(g, 'urban'),
            'wealth_index_mean':    weighted_mean(g, 'HV271') if 'HV271' in g.columns else np.nan,
            'wealth_quintile_mode': g['HV270'].mode()[0] if 'HV270' in g.columns else np.nan,
            'sample_hh':            len(g),
        })
    ).reset_index()
    
    # Fuel type breakdown
    fuel_counts = df_dhs.groupby(['county_name', 'HV226']).size().unstack(fill_value=0)
    fuel_pct    = fuel_counts.div(fuel_counts.sum(axis=1), axis=0)
    fuel_pct.columns = [f'fuel_code_{int(c)}_pct' for c in fuel_pct.columns]
    
    county_agg = county_agg.merge(fuel_pct.reset_index(), on='county_name', how='left')
    print(f'County aggregation complete: {len(county_agg)} counties')
    print(county_agg[['county_name', 'clean_cooking_rate', 'urban_rate']].sort_values(
        'clean_cooking_rate', ascending=False).head(10))
else:
    print('DHS not yet available — building with published KNCTS 2024 county estimates')
    # Use published figures from Kenya National Cooking Transition Strategy 2024
    # These are the official government baseline figures
    county_agg = pd.DataFrame({
        'county_name': list(COUNTY_CODES.values()),
    })
    print(f'Placeholder frame: {len(county_agg)} counties')

County aggregation complete: 47 counties
    county_name  clean_cooking_rate  urban_rate
29      Nairobi            0.793037    1.000000
12       Kiambu            0.631343    0.754076
9       Kajiado            0.606395    0.710807
27      Mombasa            0.445671    1.000000
43  Uasin Gishu            0.400596    0.576366
35        Nyeri            0.360138    0.242484
30       Nakuru            0.344813    0.593760
14    Kirinyaga            0.341545    0.254828
21     Machakos            0.330733    0.370184
19     Laikipia            0.281167    0.285978


---
## 2. Add Population Data & Compute Deficit

In [3]:
# Kenya 2019 Census population by county (KNBS official)
# Source: Kenya National Bureau of Statistics, 2019 Population and Housing Census
COUNTY_POPULATION_2019 = {
    'Mombasa': 1208333,    'Kwale': 866820,        'Kilifi': 1453787,
    'Tana River': 315943,  'Lamu': 143920,         'Taita Taveta': 340671,
    'Garissa': 841353,     'Wajir': 781263,         'Mandera': 867457,
    'Marsabit': 459785,    'Isiolo': 268002,        'Meru': 1545714,
    'Tharaka-Nithi': 393177, 'Embu': 608599,       'Kitui': 1136187,
    'Machakos': 1421932,   'Makueni': 987653,       'Nyandarua': 638289,
    'Nyeri': 759164,       'Kirinyaga': 610411,     "Murang'a": 1056640,
    'Kiambu': 2417735,     'Turkana': 926976,       'West Pokot': 621241,
    'Samburu': 310327,     'Trans Nzoia': 990341,   'Uasin Gishu': 1163186,
    'Elgeyo-Marakwet': 454480, 'Nandi': 885711,    'Baringo': 666763,
    'Laikipia': 518560,    'Nakuru': 2162202,       'Narok': 1157873,
    'Kajiado': 1117840,    'Kericho': 901777,       'Bomet': 875689,
    'Kakamega': 1867579,   'Vihiga': 590013,        'Bungoma': 1670570,
    'Busia': 893681,       'Siaya': 993183,         'Kisumu': 1155574,
    'Homa Bay': 1131950,   'Migori': 1116436,       'Kisii': 1266860,
    'Nyamira': 605576,     'Nairobi': 4397073
}

# Average household size Kenya 2022 DHS: 3.9 persons per HH
MEAN_HH_SIZE = 3.9

county_agg['population_2019']       = county_agg['county_name'].map(COUNTY_POPULATION_2019)
county_agg['total_households_est']  = (county_agg['population_2019'] / MEAN_HH_SIZE).round(0).astype(int)

# Core metric: households WITHOUT clean cooking
county_agg['hh_without_clean_cooking'] = (
    county_agg['total_households_est'] * (1 - county_agg['clean_cooking_rate'])
).round(0)

# Target gap: households needed to reach 100% by 2028
county_agg['hh_gap_to_100pct'] = county_agg['hh_without_clean_cooking']

# Priority score: combines gap size and access rate deficit
# Higher = more urgent (large population AND low access)
county_agg['priority_score'] = (
    county_agg['hh_without_clean_cooking'] / county_agg['hh_without_clean_cooking'].max()
) * (
    (1 - county_agg['clean_cooking_rate']) / (1 - county_agg['clean_cooking_rate']).max()
)

print('Deficit calculation complete')
print(county_agg[['county_name', 'clean_cooking_rate', 'hh_without_clean_cooking', 'priority_score']]
      .sort_values('priority_score', ascending=False).head(10))

Deficit calculation complete
   county_name  clean_cooking_rate  hh_without_clean_cooking  priority_score
10    Kakamega            0.135427                  414015.0        0.876575
2      Bungoma            0.117637                  377961.0        0.816705
25        Meru            0.133031                  343612.0        0.729530
13      Kilifi            0.140882                  320250.0        0.673772
7     Homa Bay            0.090253                  264049.0        0.588269
30      Nakuru            0.344813                  363243.0        0.582819
26      Migori            0.099928                  257660.0        0.567931
17       Kitui            0.129420                  253626.0        0.540722
42     Turkana            0.039167                  228376.0        0.537365
32       Narok            0.145209                  253780.0        0.531237


---
## 3. Merge with Shapefile → GeoDataFrame

In [4]:
shapefile_path = SHAPES / 'gadm41_KEN_1.shp'

if shapefile_path.exists():
    gdf_counties = gpd.read_file(shapefile_path)
    gdf_counties = gdf_counties.rename(columns={'NAME_1': 'county_name'})
    gdf_counties = gdf_counties[['county_name', 'geometry']]
    
    # Standardize county names for merge
    gdf_counties['county_name'] = gdf_counties['county_name'].str.strip().str.title()
    county_agg['county_name']   = county_agg['county_name'].str.strip().str.title()
    
    gdf_merged = gdf_counties.merge(county_agg, on='county_name', how='left')
    
    print(f'Merged GeoDataFrame: {len(gdf_merged)} counties')
    print(f'Missing values per column:\n{gdf_merged.isnull().sum()[gdf_merged.isnull().sum() > 0]}')
    
    # Reproject to WGS84 for web mapping
    gdf_merged = gdf_merged.to_crs(epsg=4326)
    
    # Save as GeoPackage (single file, no .shp sidecar files)
    out_path = PROCESSED / 'kenya_county_clean_cooking.gpkg'
    gdf_merged.to_file(out_path, driver='GPKG')
    print(f'\nSaved: {out_path}')
    
    # Also save flat CSV for non-GIS analysis
    gdf_merged.drop(columns='geometry').to_csv(
        PROCESSED / 'kenya_county_clean_cooking.csv', index=False
    )
    print('Saved flat CSV too.')
else:
    print('⚠️  Shapefile not found. Download from GADM first (see notebook 01).')
    # Still save the attribute table
    county_agg.to_csv(PROCESSED / 'kenya_county_clean_cooking.csv', index=False)
    print('Saved attribute CSV (no geometry).')

Merged GeoDataFrame: 47 counties
Missing values per column:
Series([], dtype: int64)

Saved: C:\Users\eric.njiraini\Documents\My Projects at a Go\kenya-clean-cooking-gap\data\processed\kenya_county_clean_cooking.gpkg
Saved flat CSV too.


---
## 4. Processing Summary

In [5]:
if 'gdf_merged' in dir():
    df = gdf_merged.drop(columns='geometry')
else:
    df = county_agg

print('=== KENYA CLEAN COOKING — COUNTY SUMMARY ===')
print(f'Total counties:                {len(df)}')
print(f'Total population (2019):       {df["population_2019"].sum():,.0f}')
print(f'Estimated total households:    {df["total_households_est"].sum():,.0f}')
print(f'HH without clean cooking:      {df["hh_without_clean_cooking"].sum():,.0f}')
print(f'Weighted avg access rate:      {(df["clean_cooking_rate"] * df["population_2019"]).sum() / df["population_2019"].sum():.1%}')
print(f'\nTop 5 counties by deficit:')
print(df.nlargest(5, 'hh_without_clean_cooking')[
    ['county_name', 'clean_cooking_rate', 'hh_without_clean_cooking']
].to_string(index=False))
print(f'\nTop 5 counties by access rate:')
print(df.nlargest(5, 'clean_cooking_rate')[
    ['county_name', 'clean_cooking_rate', 'hh_without_clean_cooking']
].to_string(index=False))

=== KENYA CLEAN COOKING — COUNTY SUMMARY ===
Total counties:                47
Total population (2019):       47,564,296
Estimated total households:    12,195,977
HH without clean cooking:      9,027,673
Weighted avg access rate:      26.0%

Top 5 counties by deficit:
county_name  clean_cooking_rate  hh_without_clean_cooking
   Kakamega            0.135427                  414015.0
    Bungoma            0.117637                  377961.0
     Nakuru            0.344813                  363243.0
       Meru            0.133031                  343612.0
     Kilifi            0.140882                  320250.0

Top 5 counties by access rate:
county_name  clean_cooking_rate  hh_without_clean_cooking
    Nairobi            0.793037                  233341.0
     Kiambu            0.631343                  228542.0
    Kajiado            0.606395                  112817.0
    Mombasa            0.445671                  171747.0
Uasin Gishu            0.400596                  178774.0
